# P&P Feature Ablation at k=1

Does **P&P uncertainty** add predictive signal beyond the **action vector** for early failure detection?

We train the same classifiers on three feature sets at cutoff **k=1** (chunks 0 and 1):

| Feature set | Description | Dim |
|---|---|---|
| `action_only` | per-dim mean action `a_mean_vec` | 14 |
| `uncertainty_only` | per-dim P&P uncertainty `u_vec` | 14 |
| `both` | concatenated u + a (original notebook) | 28 |

**Models:** Logistic Regression, Random Forest, GBM, MLP  
**Evaluation:** stratified 5-fold CV, OOF predictions, F1-optimal threshold, bootstrap AUC CIs

---

### Data provenance (must match `failure_classifier.ipynb` exactly)

| | |
|---|---|
| **Benchmark** | **LIBERO-PRO** (custom perturbed suites), not standard LIBERO |
| **Policy** | **π0.5** — `lerobot/pi05_libero_finetuned` (not SmolVLA) |
| **Episodes** | 6 suites × 10 tasks × 10 init states = **600** |
| **P&P phase used** | `pnp_mode='uncertainty'` only (`PHASE='uncertainty'`) |
| **P&P config at collection** | `step_indices=(3,4)`, `num_iterations=3`, `num_inference_steps=10` |
| **Database** | `rollouts_jeff_dimensional.db` |
| **Default path** | `MyDrive/cs159_jeff/libero_pro_results/` (Jeff's Colab Drive) |
| **Rollout source notebook** | `pnp_pro_analysis_dimensions.ipynb` (collects data; DB has ~1800 rows across 3 phases, classifier uses the 600 uncertainty rows) |

**Suites:** `libero_object_temp_x0.1`, `libero_object_temp_y0.1`, `libero_object_temp_x0.2`, `libero_object_temp_y0.2`, `libero_spatial_with_milk`, `libero_goal_with_yellow_book`

**Tables read:** `rollouts` (labels), `pnp_action_vectors` (`u_vec`, `a_mean_vec` per chunk×euler step)

This is **not** the same data as Jennifer's `results_v2/rollouts_v2.db` (π0.5 v2 slice) or `results_smolvla/rollouts_smolvla.db`.

**Outputs** (CSVs, plots) go to `MyDrive/cs159_failure_classifier/` — read-only access to Jeff's DB; nothing is written under `cs159_jeff/`.

**Reading the DB in Colab:** If `cs159_jeff` is shared with you, add a shortcut to My Drive (Drive web UI → right-click folder → *Add shortcut to Drive*). After mounting, it usually appears as `/content/drive/MyDrive/cs159_jeff/`. Alternatively copy `rollouts_jeff_dimensional.db` into `MyDrive/cs159_failure_classifier/`.

## 0. Config & paths

In [ ]:
import os, sqlite3, json as _json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score, precision_recall_curve, roc_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from IPython.display import display

# ── Path resolution (Colab Drive, shared folders, or local override) ────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive' if IN_COLAB else os.path.expanduser('~')
OUT_DIR = os.environ.get(
    'CS159_OUT_DIR',
    os.path.join(MYDRIVE, 'cs159_failure_classifier'),
)

def _find_shared_folder(name):
    """Resolve a Drive folder by name (direct My Drive path or shortcut)."""
    direct = os.path.join(MYDRIVE, name)
    if os.path.isdir(direct):
        return direct
    shortcuts = '/content/drive/.shortcut-targets-by-id'
    if os.path.isdir(shortcuts):
        for entry in os.listdir(shortcuts):
            candidate = os.path.join(shortcuts, entry, name)
            if os.path.isdir(candidate):
                return candidate
    return None

def _resolve_db_path():
    if os.environ.get('CS159_DB_PATH'):
        return os.environ['CS159_DB_PATH']
    candidates = []
    if os.environ.get('CS159_RESULTS_DIR'):
        candidates.append(os.path.join(os.environ['CS159_RESULTS_DIR'], 'rollouts_jeff_dimensional.db'))
    jeff = _find_shared_folder('cs159_jeff')
    if jeff:
        candidates.append(os.path.join(jeff, 'libero_pro_results', 'rollouts_jeff_dimensional.db'))
    candidates.extend([
        os.path.join(MYDRIVE, 'cs159_jeff', 'libero_pro_results', 'rollouts_jeff_dimensional.db'),
        os.path.join(OUT_DIR, 'rollouts_jeff_dimensional.db'),
    ])
    for path in candidates:
        if path and os.path.isfile(path):
            return path
    return candidates[0] if candidates else os.path.join(OUT_DIR, 'rollouts_jeff_dimensional.db')

DB_PATH = _resolve_db_path()
SAVE_DIR = OUT_DIR
if not os.path.exists(DB_PATH):
    tried = '\n  '.join([
        os.environ.get('CS159_DB_PATH', '(CS159_DB_PATH not set)'),
        os.path.join(os.environ.get('CS159_RESULTS_DIR', ''), 'rollouts_jeff_dimensional.db'),
        f'{MYDRIVE}/cs159_jeff/libero_pro_results/rollouts_jeff_dimensional.db',
        f'{OUT_DIR}/rollouts_jeff_dimensional.db',
    ])
    raise FileNotFoundError(
        f'Database not found.\n'
        f'Tried:\n  {tried}\n\n'
        'Fix: add a My Drive shortcut to cs159_jeff, copy the .db into cs159_failure_classifier/, '
        'or set CS159_DB_PATH to the full path.'
    )

PHASE    = 'uncertainty'
K        = 1
N_FOLDS  = 5
N_EPOCHS = 300
LR, WD   = 1e-3, 1e-4
N_BOOT   = 2000
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DIM_NAMES = ['x', 'y', 'z', 'roll', 'pitch', 'yaw', 'gripper']
FEATURE_SETS = ['action_only', 'uncertainty_only', 'both']

os.makedirs(SAVE_DIR, exist_ok=True)

print(f'OUT:    {SAVE_DIR}  (ablation outputs only — never writes to cs159_jeff)')
print(f'DB:     {DB_PATH}  (read-only)')

# ── Verify this is the same slice failure_classifier.ipynb uses ───────────────
_con = sqlite3.connect(DB_PATH)
_n_unc = _con.execute(
    "SELECT COUNT(*) FROM rollouts WHERE pnp_enabled=1 AND pnp_mode=?", (PHASE,)
).fetchone()[0]
_n_vec = _con.execute("SELECT COUNT(DISTINCT rollout_id) FROM pnp_action_vectors").fetchone()[0]
_sample = pd.read_sql(
    "SELECT suite, COUNT(*) AS n FROM rollouts "
    "WHERE pnp_enabled=1 AND pnp_mode=? GROUP BY suite ORDER BY suite",
    _con, params=(PHASE,))
_con.close()

print(f'Device: {DEVICE}')
print(f'Uncertainty rollouts: {_n_unc}  (expect 600)')
print(f'Rollouts with vector data: {_n_vec}')
if _n_unc != 600:
    print('WARNING: rollout count != 600 — may not match failure_classifier.ipynb')
display(_sample)

## 1. Load & prepare episode chunks

In [ ]:
con = sqlite3.connect(DB_PATH)
roll = pd.read_sql(
    f"SELECT rollout_id, success FROM rollouts "
    f"WHERE pnp_enabled=1 AND pnp_mode='{PHASE}'", con)
vecs = pd.read_sql(
    "SELECT rollout_id, chunk_idx, u_vec, a_mean_vec FROM pnp_action_vectors", con)
con.close()

vecs['u_vec']      = vecs['u_vec'].apply(_json.loads)
vecs['a_mean_vec'] = vecs['a_mean_vec'].apply(_json.loads)

chunk_feats = (
    vecs.groupby(['rollout_id', 'chunk_idx'], group_keys=False)
        .apply(lambda g: pd.Series({
            'u_vec':      np.mean(np.stack(g['u_vec'].values),      axis=0),
            'a_mean_vec': np.mean(np.stack(g['a_mean_vec'].values), axis=0),
        }))
        .reset_index()
)

ep_chunks = (
    chunk_feats.groupby('rollout_id', group_keys=False)
        .apply(lambda g: g.sort_values('chunk_idx')[['u_vec', 'a_mean_vec']].values.tolist())
        .reset_index()
        .rename(columns={0: 'chunks'})
        .merge(roll, on='rollout_id')
        .reset_index(drop=True)
)

n_ep   = len(ep_chunks)
n_fail = int((1 - ep_chunks['success']).sum())
print(f'Episodes: {n_ep}  ({n_fail} failures = {n_fail/n_ep:.1%})')
print(f'Chunks/episode: min={ep_chunks.chunks.map(len).min()}  '
      f'max={ep_chunks.chunks.map(len).max()}  '
      f'mean={ep_chunks.chunks.map(len).mean():.1f}')

## 2. Feature construction (k=1 ablation)

In [ ]:
def feat_names(k, mode):
    names = []
    for j in range(k + 1):
        if mode in ('uncertainty_only', 'both'):
            for d in DIM_NAMES:
                names.append(f'u_{d}_c{j}')
        if mode in ('action_only', 'both'):
            for d in DIM_NAMES:
                names.append(f'a_{d}_c{j}')
    return names


def build_Xy(k, mode='both'):
    """Build feature matrix for a given cutoff k and feature mode."""
    X, y, rids = [], [], []
    for _, r in ep_chunks.iterrows():
        if len(r['chunks']) < k + 1:
            continue
        feats = []
        for j in range(k + 1):
            u_j, a_j = r['chunks'][j]
            if mode in ('uncertainty_only', 'both'):
                feats.extend(u_j)
            if mode in ('action_only', 'both'):
                feats.extend(a_j)
        X.append(feats)
        y.append(1 - r['success'])
        rids.append(r['rollout_id'])
    return np.array(X, np.float32), np.array(y, np.float32), rids


datasets = {}
for mode in FEATURE_SETS:
    X, y, rids = build_Xy(K, mode)
    datasets[mode] = dict(X=X, y=y, rids=rids, names=feat_names(K, mode))
    print(f'{mode:18s}  shape={X.shape}  failures={y.sum():.0f}/{len(y)}')

# Sanity: all modes should use the same episodes
assert len({len(d['rids']) for d in datasets.values()}) == 1
assert all(datasets[m]['y'].tolist() == datasets['both']['y'].tolist() for m in FEATURE_SETS)

## 3. CV helpers & models

In [ ]:
def oversample(X, y):
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    if len(pos) >= len(neg):
        return X, y
    pos_up = resample(pos, n_samples=len(neg), replace=True, random_state=42)
    idx = np.concatenate([neg, pos_up])
    return X[idx], y[idx]


def best_threshold(y_true, y_prob):
    p, r, thrs = precision_recall_curve(y_true, y_prob)
    f1s = 2 * p * r / np.clip(p + r, 1e-9, None)
    return float(thrs[np.argmax(f1s[:-1])])


def eval_metrics(y_true, y_prob, thr):
    yp = (y_prob >= thr).astype(int)
    return dict(
        auc=roc_auc_score(y_true, y_prob),
        ap=average_precision_score(y_true, y_prob),
        acc=accuracy_score(y_true, yp),
        prec=precision_score(y_true, yp, zero_division=0),
        rec=recall_score(y_true, yp, zero_division=0),
        f1=f1_score(y_true, yp, zero_division=0),
        cm=confusion_matrix(y_true, yp),
    )


class FailureMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp_cv(X, y):
    if y.sum() < N_FOLDS or (1 - y).sum() < N_FOLDS:
        return None
    pw = torch.tensor([(1 - y).sum() / max(y.sum(), 1)], device=DEVICE)
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
    oof_prob = np.zeros(len(y))

    for tr, val in skf.split(X, y):
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(X[tr])
        Xvl_s = sc.transform(X[val])
        Xtr_b, ytr_b = oversample(Xtr_s, y[tr])
        Xtr_t = torch.tensor(Xtr_b, dtype=torch.float32).to(DEVICE)
        ytr_t = torch.tensor(ytr_b, dtype=torch.float32).to(DEVICE)
        Xvl_t = torch.tensor(Xvl_s, dtype=torch.float32).to(DEVICE)

        model = FailureMLP(X.shape[1]).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        crit = nn.BCEWithLogitsLoss(pos_weight=pw)
        dl = DataLoader(
            TensorDataset(Xtr_t, ytr_t),
            batch_size=min(64, len(Xtr_t)), shuffle=True,
        )

        for _ in range(N_EPOCHS):
            model.train()
            for xb, yb in dl:
                opt.zero_grad()
                loss = crit(model(xb), yb)
                loss.backward()
                opt.step()

        model.eval()
        with torch.no_grad():
            oof_prob[val] = torch.sigmoid(model(Xvl_t)).cpu().numpy()

    thr = best_threshold(y, oof_prob)
    return dict(m=eval_metrics(y, oof_prob, thr), oof=oof_prob, thr=thr)


def classical_cv(X, y, make_model):
    if y.sum() < N_FOLDS or (1 - y).sum() < N_FOLDS:
        return None
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
    oof_prob = np.zeros(len(y))
    imp_list = []

    for tr, val in skf.split(X, y):
        pipe = Pipeline([('sc', StandardScaler()), ('m', make_model())])
        Xtr_b, ytr_b = oversample(X[tr], y[tr])
        pipe.fit(Xtr_b, ytr_b)
        oof_prob[val] = pipe.predict_proba(X[val])[:, 1]
        m = pipe.named_steps['m']
        if hasattr(m, 'feature_importances_'):
            imp_list.append(m.feature_importances_)
        elif hasattr(m, 'coef_'):
            imp_list.append(np.abs(m.coef_[0]))

    thr = best_threshold(y, oof_prob)
    return dict(
        m=eval_metrics(y, oof_prob, thr), oof=oof_prob, thr=thr,
        imp=np.mean(imp_list, axis=0) if imp_list else None,
    )


MODEL_SPECS = [
    ('LogReg', lambda: LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')),
    ('RF', lambda: RandomForestClassifier(
        200, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('GBM', lambda: GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)),
    ('MLP', None),
]
FEAT_COLORS = {
    'action_only': '#3498db',
    'uncertainty_only': '#e74c3c',
    'both': '#2ecc71',
}

## 4. Train all feature sets × models

In [ ]:
results = {fs: {} for fs in FEATURE_SETS}

for fs in FEATURE_SETS:
    X, y = datasets[fs]['X'], datasets[fs]['y']
    print(f'\n── {fs} (k={K}, d={X.shape[1]}) ' + '─' * 30)
    for name, mfn in MODEL_SPECS:
        if name == 'MLP':
            r = train_mlp_cv(X, y)
        else:
            r = classical_cv(X, y, mfn)
        results[fs][name] = r
        m = r['m']
        print(f'  {name:6s}: AUC={m["auc"]:.3f}  AP={m["ap"]:.3f}  '
              f'F1={m["f1"]:.3f}  P={m["prec"]:.3f}  R={m["rec"]:.3f}')

print('\nDone.')

## 5. Summary table

In [ ]:
rows = []
for fs in FEATURE_SETS:
    for model in [s[0] for s in MODEL_SPECS]:
        m = results[fs][model]['m']
        rows.append(dict(
            feature_set=fs, model=model,
            auc=m['auc'], ap=m['ap'], f1=m['f1'],
            precision=m['prec'], recall=m['rec'], accuracy=m['acc'],
        ))

summary_df = pd.DataFrame(rows)
display(summary_df.round(3))

pivot_auc = summary_df.pivot(index='model', columns='feature_set', values='auc')
pivot_auc['Δ(both−action)'] = pivot_auc['both'] - pivot_auc['action_only']
pivot_auc['Δ(both−uncertainty)'] = pivot_auc['both'] - pivot_auc['uncertainty_only']
pivot_auc['Δ(uncertainty−action)'] = pivot_auc['uncertainty_only'] - pivot_auc['action_only']

print('\nAUC pivot (higher = better failure detection):')
display(pivot_auc.round(3))

summary_df.to_csv(os.path.join(SAVE_DIR, 'ablation_k1_summary.csv'), index=False)
pivot_auc.to_csv(os.path.join(SAVE_DIR, 'ablation_k1_auc_pivot.csv'))
print(f'\nSaved CSVs to {SAVE_DIR}')

## 6. Bootstrap AUC difference tests

Paired bootstrap on OOF predictions: is the AUC gain from adding uncertainty (or using both) statistically meaningful?

In [ ]:
def bootstrap_auc_diff(y, prob_a, prob_b, n_boot=N_BOOT, seed=42):
    """Return mean ΔAUC (b−a), 95% CI, and two-sided p-value."""
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        auc_a = roc_auc_score(y[idx], prob_a[idx])
        auc_b = roc_auc_score(y[idx], prob_b[idx])
        diffs.append(auc_b - auc_a)
    diffs = np.array(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))
    return dict(mean=float(diffs.mean()), ci_lo=float(ci_lo), ci_hi=float(ci_hi), p=float(p))


COMPARISONS = [
    ('both', 'action_only', 'both vs action_only'),
    ('both', 'uncertainty_only', 'both vs uncertainty_only'),
    ('uncertainty_only', 'action_only', 'uncertainty_only vs action_only'),
]

boot_rows = []
y = datasets['both']['y']
for model in [s[0] for s in MODEL_SPECS]:
    for a, b, label in COMPARISONS:
        pa = results[a][model]['oof']
        pb = results[b][model]['oof']
        stat = bootstrap_auc_diff(y, pa, pb)
        boot_rows.append(dict(model=model, comparison=label, **stat))

boot_df = pd.DataFrame(boot_rows)
display(boot_df.round(4))
boot_df.to_csv(os.path.join(SAVE_DIR, 'ablation_k1_bootstrap.csv'), index=False)

## 7. Visualizations

In [ ]:
models = [s[0] for s in MODEL_SPECS]
x = np.arange(len(models))
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── AUC bar chart ──
ax = axes[0]
for i, fs in enumerate(FEATURE_SETS):
    vals = [results[fs][m]['m']['auc'] for m in models]
    ax.bar(x + (i - 1) * width, vals, width, label=fs.replace('_', ' '),
           color=FEAT_COLORS[fs], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('OOF AUC')
ax.set_title('AUC by feature set (k=1)', fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=8)
ax.axhline(0.5, color='gray', ls='--', lw=0.8)

# ── ΔAUC (both − action) ──
ax = axes[1]
deltas = [results['both'][m]['m']['auc'] - results['action_only'][m]['m']['auc'] for m in models]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas]
ax.bar(models, deltas, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('ΔAUC (both − action)')
ax.set_title('Marginal value of uncertainty\n(given action features)', fontweight='bold')

# ── ΔAUC (both − uncertainty) ──
ax = axes[2]
deltas = [results['both'][m]['m']['auc'] - results['uncertainty_only'][m]['m']['auc'] for m in models]
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas]
ax.bar(models, deltas, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('ΔAUC (both − uncertainty)')
ax.set_title('Marginal value of action\n(given uncertainty features)', fontweight='bold')

plt.tight_layout()
out = os.path.join(SAVE_DIR, 'ablation_k1_bars.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {out}')

In [ ]:
y = datasets['both']['y']

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, model in zip(axes.ravel(), models):
    for fs in FEATURE_SETS:
        prob = results[fs][model]['oof']
        fpr, tpr, _ = roc_curve(y, prob)
        auc = results[fs][model]['m']['auc']
        ax.plot(fpr, tpr, lw=2, color=FEAT_COLORS[fs],
                label=f"{fs.replace('_', ' ')} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=7)

fig.suptitle('ROC curves — OOF predictions at k=1', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(SAVE_DIR, 'ablation_k1_roc.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, model in zip(axes.ravel(), models):
    for fs in FEATURE_SETS:
        prob = results[fs][model]['oof']
        prec, rec, _ = precision_recall_curve(y, prob)
        ap = results[fs][model]['m']['ap']
        ax.plot(rec, prec, lw=2, color=FEAT_COLORS[fs],
                label=f"{fs.replace('_', ' ')} ({ap:.3f})")
    ax.axhline(y.mean(), color='gray', ls='--', lw=0.8, label='base rate')
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend(fontsize=7)

fig.suptitle('Precision–Recall curves — OOF at k=1', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(SAVE_DIR, 'ablation_k1_pr.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importances for tree models (RF, GBM)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, model in enumerate(['RF', 'GBM']):
    for col, fs in enumerate(FEATURE_SETS):
        ax = axes[row, col]
        imp = results[fs][model]['imp']
        names = datasets[fs]['names']
        if imp is None:
            ax.set_visible(False)
            continue
        order = np.argsort(imp)[::-1][:10]
        ax.barh([names[i] for i in order][::-1], imp[order][::-1],
                color=FEAT_COLORS[fs])
        ax.set_title(f'{model} — {fs}', fontsize=9, fontweight='bold')
        ax.tick_params(axis='y', labelsize=7)

fig.suptitle('Top-10 feature importances (RF / GBM)', fontweight='bold', y=1.01)
plt.tight_layout()
out = os.path.join(SAVE_DIR, 'ablation_k1_importance.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

## 8. Automatic interpretation

In [ ]:
def verdict_line(model):
    auc_a = results['action_only'][model]['m']['auc']
    auc_u = results['uncertainty_only'][model]['m']['auc']
    auc_b = results['both'][model]['m']['auc']
    d_ba = bootstrap_auc_diff(y, results['action_only'][model]['oof'],
                              results['both'][model]['oof'])
    d_bu = bootstrap_auc_diff(y, results['uncertainty_only'][model]['oof'],
                              results['both'][model]['oof'])
    d_ua = bootstrap_auc_diff(y, results['action_only'][model]['oof'],
                              results['uncertainty_only'][model]['oof'])

    lines = [f'\n=== {model} ===']
    lines.append(f'  AUC  action={auc_a:.3f}  uncertainty={auc_u:.3f}  both={auc_b:.3f}')
    lines.append(
        f'  both vs action:      Δ={d_ba["mean"]:+.3f}  '
        f'95% CI [{d_ba["ci_lo"]:+.3f}, {d_ba["ci_hi"]:+.3f}]  p={d_ba["p"]:.3f}'
    )
    lines.append(
        f'  both vs uncertainty: Δ={d_bu["mean"]:+.3f}  '
        f'95% CI [{d_bu["ci_lo"]:+.3f}, {d_bu["ci_hi"]:+.3f}]  p={d_bu["p"]:.3f}'
    )
    lines.append(
        f'  uncertainty vs action: Δ={d_ua["mean"]:+.3f}  '
        f'95% CI [{d_ua["ci_lo"]:+.3f}, {d_ua["ci_hi"]:+.3f}]  p={d_ua["p"]:.3f}'
    )

    # Marginal value of uncertainty beyond action
    if d_ba['ci_lo'] > 0:
        unc_msg = 'Uncertainty ADDS significant info beyond action.'
    elif d_ba['ci_hi'] < 0:
        unc_msg = 'Uncertainty HURTS vs action alone (redundant/noisy).'
    else:
        unc_msg = 'Uncertainty does NOT significantly change AUC vs action alone.'

    # Which single modality is stronger?
    if d_ua['ci_lo'] > 0:
        cmp_msg = 'Uncertainty-only beats action-only.'
    elif d_ua['ci_hi'] < 0:
        cmp_msg = 'Action-only beats uncertainty-only.'
    else:
        cmp_msg = 'Action and uncertainty are statistically tied alone.'

    lines.append(f'  → {unc_msg}')
    lines.append(f'  → {cmp_msg}')
    return '\n'.join(lines)


print('=' * 72)
print('ABLATION VERDICT at k=1')
print('=' * 72)
for model in models:
    print(verdict_line(model))

# Aggregate across models
mean_d_ba = pivot_auc['Δ(both−action)'].mean()
mean_d_ua = pivot_auc['Δ(uncertainty−action)'].mean()
best_single = 'uncertainty' if pivot_auc['uncertainty_only'].mean() > pivot_auc['action_only'].mean() else 'action'
best_both_gain = pivot_auc['Δ(both−action)'].max()
best_both_model = pivot_auc['Δ(both−action)'].idxmax()

print('\n' + '=' * 72)
print('AGGREGATE (across models)')
print('=' * 72)
print(f'  Mean AUC — action: {pivot_auc["action_only"].mean():.3f}  '
      f'uncertainty: {pivot_auc["uncertainty_only"].mean():.3f}  '
      f'both: {pivot_auc["both"].mean():.3f}')
print(f'  Mean Δ(both−action): {mean_d_ba:+.3f}')
print(f'  Mean Δ(uncertainty−action): {mean_d_ua:+.3f}')
print(f'  Stronger single modality on average: {best_single}')
print(f'  Largest marginal gain from combining: {best_both_model} (+{best_both_gain:.3f} AUC)')

if mean_d_ba > 0.01 and (boot_df[(boot_df.comparison == 'both vs action_only')]['ci_lo'] > 0).any():
    headline = (
        'YES — P&P uncertainty adds information beyond the action vector for at least one model.'
    )
elif mean_d_ba < -0.005:
    headline = (
        'NO — combining uncertainty does not help; action features alone are sufficient or better.'
    )
else:
    headline = (
        'INCONCLUSIVE — uncertainty provides little marginal AUC gain beyond action at k=1.'
    )
print(f'\n  HEADLINE: {headline}')
print('=' * 72)